# Leaf stomatal efficiency
 

In [ ]:
# | default_exp leaf_stomatal_efficiency

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
from plant_hydraulics.leaf_temperature import leaf_temperature
from plant_hydraulics.leaf_boundary_layer import leaf_boundary_layer
from plant_hydraulics.leaf_photosynthesis import leaf_photosynthesis
from plant_hydraulics.leaf_water_potential import leaf_water_potential
from plant_hydraulics.parameter_classes import PhysCon, Leaf, Flux, Atmos

```
Input: gs_val (candidate stomatal conductance)
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Compute boundary layer conductances │
    └──────────────────────────────────────┘
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
At gs = gs_val − delta    At gs = gs_val
   ┌───────────────┐    ┌───────────────┐
   │ Leaf temp     │    │ Leaf temp     │
   │ Photosynthesis│    │ Photosynthesis│
   │ → an2         │    │ → an1         │
   └───────────────┘    └───────────────┘
          │                   │
          └─────────┬─────────┘
                    ▼
    ┌───────────────────────────────────────┐
    │  CHECK 1: Efficiency                  │
    │wue = (an1 − an2) − delta·iota·(VPD/P) │
    │                                       │
    │  Marginal carbon  vs  Marginal water  │
    │  gain                    cost         │
    │                                       │
    │       wue > 0 → "still worth it"      │
    │       wue < 0 → "too expensive"       │
    └───────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  CHECK 2: Cavitation safety          │
    │  leafwp = leaf_water_potential()     │
    │  minpsi = leafwp − ψ_min             │
    │                                      │
    │  minpsi > 0 → "No cavitation risk"   │
    │  minpsi < 0 → "danger of cavitation" │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  val = min(wue, minpsi)              │
    │                                      │
    │  val > 0 → open stomata more         │
    │  val < 0 → close stomata             │
    │  val = 0 → OPTIMAL gs (root)         │
    └──────────────────────────────────────┘
                    │
                    ▼
          Return val to Brent's method
```

In [ ]:
# | export


def leaf_stomatal_efficiency(
    physcon: PhysCon, atmos: Atmos, leaf: Leaf, flux: Flux, gs_val: float
) -> tuple[Flux, float]:
    """
    Stomatal water-use efficiency and cavitation check to determine
    maximum gs.

    For the stomatal conductance gs_val, calculate photosynthesis and leaf
    water potential for an increase in stomatal conductance equal to "delta".

    The returned value is positive if this increase produces a change in
    photosynthesis > iota*vpd*delta or if the leaf water potential is > minl_wp.

    The returned value is negative if the increase produces a change in
    photosynthesis < iota*vpd*delta or if the leaf water potential is < minl_wp.

    """
    # Leaf boundary layer conductances ------------------------------------------
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)

    # Specify "delta" as a small difference in gs (mol H2O/m2/s) ----------------
    # 0.001 mol/m2/s
    delta = 0.001

    # Calculate photosynthesis at two slightly different stomatal openings ------

    # Photosynthesis at lower gs (gs_val - delta)
    # Calculate photosynthesis at gs_val - delta, but first need leaf
    # temperature for this gs
    
    # gs2 =  lower value for gs (mol H2O/m2/s)
    gs2 = gs_val - delta
    flux.gs = gs2
    flux = leaf_temperature(physcon, atmos, leaf, flux)
    flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
    
    # an2 = leaf photosynthesis at gs2 (umol CO2/m2/s)
    an2 = flux.an

    # Photosynthesis at higher gs     
    # Calculate photosynthesis at higher gs (gs_val)
    
    # gs1 - higher value for gs (mol H2O/m2/s)
    gs1 = gs_val
    flux.gs = gs1
    flux = leaf_temperature(physcon, atmos, leaf, flux)
    flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
    
    # an1 - leaf photosynthesis at gs1 (umol CO2/m2/s)
    an1 = flux.an

    # Efficiency check ----------------------------------------------------------
    
    # 1) (an1 - an2) -> This represents the extra carbon gained by opening 
    # stomata by a little bit (i.e delta)
    #
    # 2) (flux.vpd / atmos.patm) -> This converts stomatal conductance to an
    # actual water vapour flux (the drier the air, the more water you lose per
    # unit of stomatal opening)
    #
    # 3) (leaf.iota) -> Represents how much carbon the plant considers each mole
    # of water to be "worth". Units µmol CO2 / mol H2O. In the book, leaf.iota
    # represents the lagrange multiplier
    
    # Examples:
    # 
    # A plant in a desert (water is scarce) would have a high iota, it demands a 
    # lot of carbon return for every drop of water invested
    #
    # A plant in a rainforest (water is abundant) would have a low iota, it 
    # doesn't mind spending water freely.
    #
    # Overall, The (leaf.iota * delta * (flux.vpd / atmos.patm)) represents
    # the water cost of that extra opening, expressed in "carbon-equivalent"
    #
    # Is wue < 0 when d(An)/d(gs) < iota * vpd?
    #
    # When wue > 0: the extra carbon gained exceeds the water cost "keep opening,
    # it's profitable!"
    #
    # When wue < 0: the extra carbon gained is less than the water cost → "stop,
    # water is being wasted"
    #
    # When wue = 0: the marginal carbon gain exactly equals the marginal water
    # cost → optimal gs.
    wue = (an1 - an2) - leaf.iota * delta * (flux.vpd / atmos.patm)

    # Cavitation check ----------------------------------------------------------
    # Is minpsi < 0 when leaf_wp < minl_wp ?
    # When minpsi > 0: ψ_leaf is safely above the minimum
    # When minpsi < 0: ψ_leaf has dropped below the danger threshold →
    # "close stomata"
    flux = leaf_water_potential(physcon, leaf, flux)
    min_psi = flux.psi_leaf - leaf.minl_wp

    # Return the minimum of the two checks --------------------------------------
    # Returns whichever check is more negative. 
    val = min(wue, min_psi)

    return flux, val

#### Example leaf_stomatal_efficiency()